In [2]:
import pandas as pd
import pyodbc
from sqlalchemy import create_engine
import json
import urllib
import datetime

# ==========================================
# 1. CONFIGURACIÓN DE CONEXIÓN Y METAS
# ==========================================
server = 'db.gabssa.app' 
database = 'BAHIA'
username = 'Usr_Canales'
password = 'Huyt&233_Trwt'

META_POR_ASESOR = 380 

params = urllib.parse.quote_plus(f"DRIVER={{SQL Server}};SERVER={server};DATABASE={database};UID={username};PWD={password};")
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

def generar_datos():
    try:
        hoy = datetime.datetime.now()
        mes, anio = hoy.strftime('%m'), hoy.strftime('%Y')
        ayer_dt = hoy - datetime.timedelta(days=1)

        print(f"[{hoy.strftime('%H:%M:%S')}] Extrayendo datos...")

        # --- 2. SQL QUERIES ---
        df_gestiones = pd.read_sql("""
            SELECT usuario_id, ini, asunto FROM [BAHIA].[dbo].[daimler_interaccion] 
            WHERE CAST(ini AS DATE) = CAST(DATEADD(DAY, -1, GETDATE()) AS DATE) AND cliente <> 'DAIMLER'
        """, engine)

        df_promesas = pd.read_sql(f"""
            SELECT i.usuario_id, CONCAT(p.[NOMBRE_GESTOR], ' ', p.[APELLIDO_PATERNO]) AS Nombre, i.importe, i.fecha_captura
            FROM [BAHIA].[dbo].[daimler_pago] i
            INNER JOIN [BAHIA].[dbo].[VW_PADRONLIGTH] p ON i.usuario_id = p.ID_EMPLEADO
            WHERE cliente <> 'DAIMLER' AND MONTH(fecha_captura) = '{mes}' AND YEAR(fecha_captura) = '{anio}'
        """, engine)

        # --- 3. PROCESAMIENTO ---
        total_g = len(df_gestiones)
        asesores_activos = max(df_gestiones['usuario_id'].nunique(), 1)
        
        # KPI Promedio y RPC%
        promedio_persona = round(total_g / asesores_activos, 0)
        
        # Definición de contacto real (Ajusta según tus estatus)
        estatus_contacto = ['PROMESA DE PAGO', 'DICE QUE YA PAGÓ', 'NEGATIVA DE PAGO', 'ACLARACIÓN', 'RECADÓ CON FAMILIAR']
        contactos_reales = df_gestiones[df_gestiones['asunto'].isin(estatus_contacto)].shape[0]
        rpc_pct = round((contactos_reales / total_g * 100), 1) if total_g > 0 else 0

        # Tiempo Muerto
        meta_grupal = asesores_activos * META_POR_ASESOR
        tiempo_muerto = max(0, round(((meta_grupal - total_g) / meta_grupal) * 100, 1)) if total_g < meta_grupal else 2.0

        carrera = df_promesas.groupby('Nombre').size().sort_values(ascending=False).head(10).reset_index(name='Cant')
        df_p_ayer = df_promesas[pd.to_datetime(df_promesas['fecha_captura']).dt.date == ayer_dt.date()]

        # --- 4. PREMIOS ---
        g_u = df_gestiones.groupby('usuario_id').size()
        p_u = df_p_ayer.groupby('usuario_id').size()
        efec_u = (p_u / g_u).dropna().sort_values(ascending=False)
        
        def get_nom(uid): return df_promesas[df_promesas['usuario_id']==uid]['Nombre'].iloc[0] if uid in df_promesas['usuario_id'].values else f"ID:{uid}"
        
        premios = {
            "efectividad": get_nom(efec_u.index[0]) if not efec_u.empty else "---",
            "trabajador": get_nom(g_u.idxmax()) if not g_u.empty else "---",
            "racha": carrera.iloc[0]['Nombre'] if not carrera.empty else "---"
        }

        # --- 5. TENDENCIAS ---
        df_gestiones['hora'] = pd.to_datetime(df_gestiones['ini']).dt.hour
        g_hora = df_gestiones.groupby('hora').size().reindex(range(8, 21), fill_value=0)
        contact = df_gestiones['asunto'].value_counts().head(8).to_dict()

        data = {
            "kpis": {
                "total_promesas_mes": int(len(df_promesas)),
                "cantidad_gestiones_ayer": total_g,
                "efectividad_ayer": round((len(df_p_ayer)/total_g*100), 2) if total_g > 0 else 0,
                "tiempo_muerto_pct": tiempo_muerto,
                "promedio_por_asesor": int(promedio_persona),
                "rpc_pct": rpc_pct
            },
            "premios": premios,
            "carrera": {"nombres": carrera['Nombre'].tolist(), "cantidades": carrera['Cant'].tolist()},
            "grafica_hora": {"horas": [f"{h}h" for h in g_hora.index], "conteo": g_hora.tolist()},
            "contactability": contact
        }

        with open('datos.js', 'w', encoding='utf-8') as f:
            f.write(f"const datosDaimler = {json.dumps(data, indent=4, ensure_ascii=False)};")
        print("✅ Dashboard actualizado correctamente.")

    except Exception as e: print(f"❌ Error: {e}")

if __name__ == "__main__": generar_datos()

[12:42:47] Extrayendo datos...
✅ Dashboard actualizado correctamente.
